# Implementing BPE Tokenizer from Scratch

> In the previous section, we saw that subword-level tokenizers strike the best balance between character-level and word-level approaches -- common words are kept intact, while rare words are split into known fragments. But we haven't yet explored the specific rules for how to split text, or how the vocabulary is built from scratch.
>
> In this section, we'll train a BPE tokenizer directly, observe its encoding and decoding behavior, and compare it with GPT-2's real tokenizer. The step-by-step derivation of the BPE algorithm is in the appendix at the end of this chapter -- if you're curious about the internal details after reading the main text, flip to the back for a closer look.

BPE (Byte Pair Encoding) has only two core operations: count which adjacent character pairs appear most frequently in the corpus, then merge the most frequent pair into a new subword unit. Repeating this process, the vocabulary gradually expands from individual characters to longer subwords.

Let's train a BPE tokenizer on 10 English sentences and see what it produces.

## 1. BPE Quick Experience

In [1]:
class BPETokenizer:
    """
    Complete BPE Tokenizer implementation

    Three core functions:
    1. train()  -- learn merge rules from corpus
    2. encode() -- text -> token ID list
    3. decode() -- token ID list -> text
    """

    def __init__(self):
        # merge_rules: records each merged pair in order
        # This is BPE's core data structure; encode applies them greedily in this order
        self.merge_rules = []
        # Final vocabulary: initial characters + new tokens learned at each merge step
        self.vocab = set()
        self.stoi = {}
        self.itos = {}

    def train(self, texts, num_merges=15, verbose=True):
        """
        BPE training

        Parameters:
            texts: list of corpus strings
            num_merges: number of merge operations
            verbose: whether to print each step
        """
        # Step 0: initialize at character level
        token_lists = [list(text) for text in texts]

        # Collect all unique characters in the corpus as the initial vocabulary
        base_vocab = set()
        for tokens in token_lists:
            for c in tokens:
                base_vocab.add(c)
        learned_vocab = set(base_vocab)

        if verbose:
            print(f"{'='*60}")
            print(f"BPE training started! Initial state: each sentence split into characters")
            print(f"{'='*60}")
            print(f"Initial character set size: {len(base_vocab)}")
            print()

        for step in range(num_merges):
            # Step 1: count frequency of all pairs
            pairs = {}
            for tokens in token_lists:
                for i in range(len(tokens) - 1):
                    pair = (tokens[i], tokens[i + 1])
                    if pair not in pairs:
                        pairs[pair] = 0
                    pairs[pair] += 1

            if not pairs:
                break

            # Step 2: find the most frequent pair
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]

            # Step 3: record the merge rule
            self.merge_rules.append(best_pair)

            # Step 4: merge
            a, b = best_pair
            new_token = a + b

            new_token_lists = []
            for tokens in token_lists:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                new_token_lists.append(new_tokens)

            token_lists = new_token_lists

            learned_vocab.add(new_token)

            if verbose:
                print(f"Step {step+1:2d}: merge {best_pair} -> '{new_token}'  (appears {best_count} times) | Current vocab size: {len(learned_vocab)}")

        # Build the final vocabulary
        # Note: the vocabulary cannot only keep tokens that still appear in the final corpus.
        # When encode encounters an unseen combination, it needs to fall back to initial characters.
        self.vocab = learned_vocab
        sorted_vocab = sorted(list(self.vocab))
        self.stoi = {t: i for i, t in enumerate(sorted_vocab)}
        self.itos = {i: t for i, t in enumerate(sorted_vocab)}

        if verbose:
            print(f"\n{'='*60}")
            print(f"Training complete!")
            print(f"  - Number of merges: {len(self.merge_rules)}")
            print(f"  - Final vocab size: {len(self.vocab)} tokens")
            print(f"  - Merge rules: {self.merge_rules}")
            print(f"{'='*60}")

        return token_lists

In [2]:
# Experiment corpus
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

# Train BPE tokenizer (without printing each step)
bpe = BPETokenizer()
final_tokens = bpe.train(corpus, num_merges=15, verbose=False)

print(f"Training complete!")
print(f"  Initial characters: {len(set(c for text in corpus for c in text))}")
print(f"  Number of merges:   {len(bpe.merge_rules)}")
print(f"  Final vocabulary:   {len(bpe.vocab)} tokens")
print()
print("Learned merge rules (first 8):")
for i, (a, b) in enumerate(bpe.merge_rules[:8]):
    print(f"  {i+1}. '{a}' + '{b}' -> '{a+b}'")

Training finished!Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.  1. 'e' + ' ' → 'e '  2. 't' + 'h' → 'th'  3. 'th' + 'e ' → 'the '  4. 'a' + 't' → 'at'  5. 'o' + 'g' → 'og'  6. 'at' + ' ' → 'at '  7. 's' + ' ' → 's '  8. 'd' + 'og' → 'dog'

### Encoding and Decoding

Once we have the merge rules from training, encode merges text into a token sequence by applying rules in order, while decode concatenates tokens back into the original text.

In [3]:
# Add the encode method to BPETokenizer
def bpe_encode(self, text):
    """
    BPE encoding: text -> token ID list

    Core logic: apply merge rules greedily in training order
    """
    # Step 1: split into characters
    tokens = list(text)

    # Step 2: apply each merge rule in order
    for (a, b) in self.merge_rules:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    # Step 3: look up each token's ID in the vocabulary.
    # If not found, the input contains a character not seen during training; do not silently skip it.
    ids = []
    for token in tokens:
        if token not in self.stoi:
            raise KeyError(token)
        ids.append(self.stoi[token])
    return ids

# Attach the method
BPETokenizer.encode = bpe_encode

print("encode method added!")

Read the values printed above and connect them to the concept in this cell.

In [4]:
# Test encode
text = "the cat"
ids = bpe.encode(text)
print(f"Original text: '{text}'")
print(f"Token IDs: {ids}")
print(f"Number of tokens: {len(ids)}")
print(f"Token-by-token breakdown:")
for i, tid in enumerate(ids):
    print(f"  ID {tid} -> '{bpe.itos[tid]}'")
print(f"Original text characters = {len(text)}, BPE token count = {len(ids)}, compression ratio = {len(text)/len(ids):.1f}x")

print()

# Decode: concatenate token IDs back to text
def bpe_decode(self, ids):
    """BPE decoding: token ID list -> text"""
    return "".join([self.itos[i] for i in ids])

BPETokenizer.decode = bpe_decode

# Test full encode-decode cycle
for test_text in ["the cat", "my dog is happy", "i love cats"]:
    ids = bpe.encode(test_text)
    recovered = bpe.decode(ids)
    status = "OK" if test_text == recovered else "FAIL"
    print(f"[{status}] '{test_text}' -> {ids} -> '{recovered}'")

Original text: 'the cat'Token IDs: [30, 3]
Number of tokens: 2Explanation: the printed values show the main mechanism in this step.  ID 30 → 'the c'
  ID 3 → 'at'
Number of tokens: 
[OK] 'the cat' → [30, 3] → 'the cat'
[OK] 'my dog is happy' → [16, 34, 0, 7, 0, 14, 12, 2, 21, 21, 34] → 'my dog is happy'
[OK] 'i love cats' → [13, 0, 15, 19, 33, 9, 5, 3, 23] → 'i love cats'


### Why BPE Handles Most Unknown Words

BPE's encode process starts from characters and merges them step by step according to merge rules. This leads to a key implication: **as long as every character in the input text is in the initial vocabulary, there will be no unknown tokens.**

Consider an example. Suppose the training corpus has seen "play" and "ing", but never "playing". When encode processes "playing", it first splits the text into characters `['p','l','a','y','i','n','g']`, then applies merge rules one by one. If the rules include merges like `('p','l')->'pl'` and `('a','y')->'ay'`, the result might be `['play', 'ing']`. Even though "playing" has never appeared as a whole, it can be split into two known subwords.

This property comes from encode's design: it doesn't re-count frequencies and merge from scratch, but strictly follows the rules learned during training. Both sides of each merge rule -- the two tokens before merging and the new token after merging -- are in the vocabulary, so the merged product will always have an ID. Looking back at encode's final step:

```python
# Step 3: look up each token's ID in the vocabulary
ids = []
for token in tokens:
    if token not in self.stoi:
        raise KeyError(token)
    ids.append(self.stoi[token])
```

There's no fallback logic here to "split unknown tokens back into characters." This isn't laziness -- it's unnecessary: encode starts from characters and only merges, never splits, so as long as the characters are in the vocabulary, the resulting tokens will always be in the vocabulary.

But note the prerequisite: all characters in the input text must have appeared in the training corpus. If it encounters a character never seen during training -- like the digit `3`, a newline, or the emoji `👽` -- the lookup will raise an error directly. The correct approach is to report the error explicitly rather than silently dropping unknown characters.

This is why industrial-grade tokenizers usually don't start from characters, but instead use Byte-Level BPE. The 256 base bytes can combine to represent any text or symbol in the world, avoiding OOV (Out-Of-Vocabulary) problems at the representation level. When faced with the never-before-seen `👽`, Byte-Level BPE doesn't treat it as an unrecognizable new character. Instead, it first converts it to 4 bytes in UTF-8 encoding: `[240, 159, 145, 189]`. Even if the system has never learned to merge them into an alien emoji, it can still reliably find these 4 individual byte IDs in the 0-255 base vocabulary.

In [5]:
# Demo: encoding a sentence not in the training corpus
unseen = "a cat runs fast"

# First verify: this sentence is indeed not in the training corpus
print(f"Test sentence: '{unseen}'")
in_corpus = any(unseen == s for s in corpus)
print(f"Appeared in training corpus: {in_corpus}")

# But all its characters are in the vocabulary
unseen_chars = set(unseen)
vocab_chars = bpe.vocab
missing = [c for c in unseen_chars if c not in vocab_chars]
print(f"Characters in sentence: {sorted(unseen_chars)}")
print(f"Characters not in vocabulary: {missing if missing else 'none'}")
print()

ids = bpe.encode(unseen)
recovered = bpe.decode(ids)

print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"Status: {'OK' if unseen == recovered else 'FAIL'}")

# See what BPE split it into
print(f"\nIndividual tokens:")
for tid in ids:
    print(f"  '{bpe.itos[tid]}'", end="")
print()

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Token IDs: [2, 0, 5, 4, 22, 32, 17, 24, 10, 2, 23, 27]
Decoded back: 'a cat runs fast'Read the values printed above and connect them to the concept in this cell.
Inspect each token:  'a'  ' '  'c'  'at '  'r'  'u'  'n'  's '  'f'  'a'  's'  't'


## 2. Comparing with GPT-2's Real Tokenizer

In [6]:
# Load the real GPT-2 tokenizer (byte-level BPE)
try:
    import tiktoken

    real_tokenizer_name = "gpt2"
    real_tokenizer = tiktoken.get_encoding(real_tokenizer_name)
    print(f"Real tokenizer: {real_tokenizer_name}")
    print(f"Vocabulary size: {real_tokenizer.n_vocab}")
except Exception as e:
    real_tokenizer = None
    print("Could not load tiktoken's GPT-2 tokenizer")
    print(f"Reason: {e}")
    print("After installing and caching tiktoken, this cell will print real tokenizer results.")


def show_real_tokenization(text):
    """Print how the real tokenizer splits text into tokens and IDs"""
    ids = real_tokenizer.encode(text)
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]

    print(f"Text: {text!r}")
    print(f"Token count: {len(ids)}")
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(real_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")
    return ids, tokens


if real_tokenizer is not None:
    show_real_tokenization("the cat sat on the mat")

a real tokenizer: gpt2Vocabulary size: 50257Text: 'the cat sat on the mat'Number of tokens:   00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
  02 | id=3332   | token=' sat' | bytes=[32, 115, 97, 116]
  03 | id=319    | token=' on' | bytes=[32, 111, 110]
  04 | id=262    | token=' the' | bytes=[32, 116, 104, 101]
  05 | id=2603   | token=' mat' | bytes=[32, 109, 97, 116]


In [7]:
def try_mini_bpe(text):
    """Encode with our mini BPE; return error reason on failure"""
    try:
        ids = bpe.encode(text)
        tokens = [bpe.itos[i] for i in ids]
        return ids, tokens, None
    except KeyError as e:
        return None, None, e.args[0]


compare_texts = [
    "the cat sat on the mat",
    " the cat",
    "the  cat",
    "327 + 1 = 328",
    "def f():\n    return x",
    "你好，世界🙂",
]

if real_tokenizer is not None:
    for text in compare_texts:
        print("=" * 68)
        print(f"Text: {text!r}")

        mini_ids, mini_tokens, error = try_mini_bpe(text)
        if error is None:
            print(f"mini BPE: {len(mini_ids)} tokens")
            print(f"  tokens: {mini_tokens}")
            print(f"  ids:    {mini_ids}")
        else:
            print("mini BPE: cannot encode")
            print(f"  Reason: character {error!r} not in small corpus vocabulary")

        real_ids = real_tokenizer.encode(text)
        real_tokens = [real_tokenizer.decode([tok_id]) for tok_id in real_ids]
        real_tokens = [t.replace("\n", "\\n") for t in real_tokens]
        print(f"Real GPT-2 BPE: {len(real_ids)} tokens")
        print(f"  tokens: {real_tokens}")
        print(f"  ids:    {real_ids}")

    print("=" * 68)
    print("Key observation: The real tokenizer is not just bigger -- it also handles bytes, spaces, newlines, and Unicode.")
    print("Our mini BPE is for understanding merge rules; the industrial version solves coverage, speed, and edge cases.")

Text: 'the cat sat on the mat'mini BPE: 6 tokens
  tokens: ['the cat ', 'sat o', 'n', ' the ', 'm', 'at']
  ids:    [31, 26, 17, 1, 16, 3]
Read the values printed above and connect them to the concept in this cell.  tokens: ['the', ' cat', ' sat', ' on', ' the', ' mat']
  ids:    [1169, 3797, 3332, 319, 262, 2603]
Text: ' the cat'mini BPE: 3 tokens
  tokens: [' ', 'the c', 'at']
  ids:    [0, 30, 3]
Read the values printed above and connect them to the concept in this cell.  tokens: [' the', ' cat']
  ids:    [262, 3797]
Text: 'the cat'mini BPE: 4 tokens
  tokens: ['the ', ' ', 'c', 'at']
  ids:    [29, 0, 5, 3]
Read the values printed above and connect them to the concept in this cell.  tokens: ['the', ' ', ' cat']
  ids:    [1169, 220, 3797]
Text: '327 + 1 = 328'Read the values printed above and connect them to the concept in this cell.ReasonRead the values printed above and connect them to the concept in this cell.  tokens: ['327', ' +', ' 1', ' =', ' 328']
  ids:    [34159, 1343, 3

### Numbers, Spaces, Indentation, and Chinese

Many "strange behaviors" of real tokenizers are related to the training corpus and engineering choices. These details directly affect how the model performs on math, code, and mixed Chinese-English text.

In [8]:
special_cases = [
    ("Numbers", "327 + 1 = 328"),
    ("Leading space", "cat"),
    ("Same word with leading space", " cat"),
    ("Multiple spaces", "the    cat"),
    ("Newlines and indentation", "def f():\n    return x"),
    ("Chinese and emoji", "你好，世界🙂"),
]

if real_tokenizer is not None:
    for label, text in special_cases:
        print("=" * 68)
        print(f"Special case: {label}")
        show_real_tokenization(text)

    print("=" * 68)
    print("Special token: <|endoftext|>")
    text = "hello<|endoftext|>world"

    try:
        real_tokenizer.encode(text)
    except ValueError as e:
        print("Default encode refuses to silently swallow special tokens as ordinary text.")
        print(f"Error summary: {str(e).splitlines()[0]}")

    ids = real_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
    print("After allowing special token:")
    print(f"  ids:    {ids}")
    print(f"  tokens: {tokens}")
    print("Key point: special tokens are extra control symbols, not learned from ordinary merge rules.")

Read the values printed above and connect them to the concept in this cell.Text: '327 + 1 = 328'Number of tokens:   00 | id=34159  | token='327' | bytes=[51, 50, 55]
  01 | id=1343   | token=' +' | bytes=[32, 43]
  02 | id=352    | token=' 1' | bytes=[32, 49]
  03 | id=796    | token=' =' | bytes=[32, 61]
  04 | id=39093  | token=' 328' | bytes=[32, 51, 50, 56]
Read the values printed above and connect them to the concept in this cell.Text: 'cat'Number of tokens:   00 | id=9246   | token='cat' | bytes=[99, 97, 116]
Read the values printed above and connect them to the concept in this cell.Text: ' cat'Number of tokens:   00 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
Read the values printed above and connect them to the concept in this cell.Text: 'the cat'Number of tokens:   00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=220    | token=' ' | bytes=[32]
  02 | id=220    | token=' ' | bytes=[32]
  03 | id=220    | token=' ' | bytes=[32]
  04 | id=3797   | token=' 

### Fun Question: Why Does the Model Sometimes Think 1.11 Is Greater Than 1.9

Before looking at the answer, try guessing for yourself:

```text
1.11 and 1.9, which is larger?
```

Mathematically, of course 1.9 is larger. But an LLM doesn't immediately treat 1.11 as a "decimal object." What it sees first are the token IDs produced by the tokenizer. If the tokenization looks like this:

```text
"1.11" -> ["1", ".", "11"]
"1.9"  -> ["1", ".", "9"]
```

The model might be misled by surface patterns: 11 looks larger than 9, so it might mistakenly think 1.11 > 1.9. This doesn't mean the model can't compare decimals at all -- it means it first needs to learn the rule of "decimal place alignment" from token sequences.

In [9]:
# Fun observation: how the real tokenizer splits 1.11 and 1.9
number_texts = ["1.11", "1.9"]

if real_tokenizer is not None:
    for text in number_texts:
        ids = real_tokenizer.encode(text)
        tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
        print(f"{text!r} -> tokens={tokens}, ids={ids}")

    print()
    print("Mathematical comparison:")
    print(f"  1.11 > 1.9  ? {1.11 > 1.9}")
    print(f"  1.90 > 1.11 ? {1.90 > 1.11}")
    print()
    print("Key observation: The model doesn't see floats -- it sees token sequences.")
    print("To compare decimals, it must learn to pad decimal places, not directly compare tokens '11' and '9'.")

'1.11' -> tokens=['1', '.', '11'], ids=[16, 13, 1157]
'1.9' -> tokens=['1', '.', '9'], ids=[16, 13, 24]

Read the values printed above and connect them to the concept in this cell.  1.11 > 1.9  ? False
  1.90 > 1.11 ? True

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

## 3. Key Design Choices in Industrial BPE

### The Impact of Vocabulary Size

More merges mean a larger vocabulary and more "complete" tokens. But bigger isn't always better -- too small a vocabulary means very long sequences; too large means many tokens rarely appear, making learning wasteful. Let's see how the same sentence is split under different merge counts:

In [10]:
# Train multiple tokenizers with different merge counts
for n_merges in [5, 15, 30]:
    t = BPETokenizer()
    t.train(corpus, num_merges=n_merges, verbose=False)

    test = "the cat sat on the mat"
    ids = t.encode(test)

    print(f"merge={n_merges:2d} | vocab={len(t.vocab):2d} | tokens={len(ids):2d} | tokens: {[t.itos[i] for i in ids]}")

print()
print("Observation: More merges means common fragments are more likely kept intact, resulting in fewer tokens.")

Number of tokens: Number of tokens: Number of tokens: 
Number of tokens: 

### Pre-tokenization: Split Words First, Then Apply BPE

Our mini BPE counts pairs across entire sentences. The result is that BPE might merge across word boundaries -- for example, `e` and a space might merge into `e `, gluing a word's ending to a space. GPT-2's approach: first use a regular expression to split text into independent fragments (roughly at the "word" level), then apply BPE to each fragment separately. This way merging only happens within words, never across word boundaries.

In [11]:
import re

# GPT-2's pre-tokenization regex
# Splits text into independent fragments by "words + contractions + punctuation + digits"
gpt2_pattern = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
)

text = "Hello, I'm learning BPE! It's really cool."

chunks = gpt2_pattern.findall(text)

print(f"Original text: '{text}'")
print(f"Pre-tokenization result:")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] '{chunk}'")

print()
print("Key observation: Each fragment is either a word (possibly with a leading space), punctuation, or a contraction.")
print("BPE only counts and merges within each fragment, never across fragments.")
print()
print("Comparison with our mini version:")
print("  Mini version: 'the cat' -> ['t','h','e',' ','c','a','t']  <- space is an ordinary character, can merge across words")
print("  Industrial version: 'the cat' -> ['the', ' cat']           <- split words first, BPE operates within words")

Original text: 'Hello, I'm learning BPE! It's really cool.'Pre-tokenization Result:  [0] 'Hello'
  [1] ','
  [2] ' I'
  [3] ''m'
  [4] ' learning'
  [5] ' BPE'
  [6] '!'
  [7] ' It'
  [8] ''s'
  [9] ' really'
  [10] ' cool'
  [11] '.'

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### Byte-Level Starting Point

A character-level starting point seems stable enough, but the Unicode world is too large. For example, the Chinese character "你" is actually 3 bytes in UTF-8: E4 BD A0; the emoji "😊" is 4 bytes: F0 9F 98 8A. If the tokenizer starts from characters, it needs to recognize all kinds of Chinese characters, emoji, and special symbols in advance. But if it starts from bytes, there are only 256 possibilities at the starting point. No matter what text is input, it can at least be split into bytes without going OOV.

| | Our version | Industrial version |
|:---|:---|:---|
| Starting point | Characters | Bytes, covering all Unicode |
| Spaces | Treated as ordinary characters | Special handling for leading spaces, newlines, indentation |
| Counting | Full frequency counting | More efficient data structures for speed |
| Corpus | Small corpus | Massive corpus |

The core idea hasn't changed: count pairs -> merge the most frequent -> repeat -> obtain ordered merge rules.

### Special Tokens

Tokens learned by BPE from the corpus are one category -- "the", "ing", "tion" -- because they appear frequently and get merged. There's another category of tokens in the vocabulary that didn't come from pair statistics, but were manually added when training the tokenizer. These are called special tokens.

| Token string | Function name | Meaning |
|:---|:---|:---|
| `<unk>` | Unknown (UNK) | Placeholder when encountering characters that cannot be represented |
| `<s>` | BOS (Beginning of Sequence) | Marks the beginning of an input sequence |
| `</s>` | EOS (End of Sequence) | Marks the end of an input sequence |

LLaMA / LLaMA 2 uses the three above. Modern chat models need finer-grained control -- Qwen uses `<|im_start|>` / `<|im_end|>` to mark message boundaries in each turn, while LLaMA 3 uses `<|begin_of_text|>` / `<|eot_id|>` to distinguish between full documents and individual conversation turns.

Adding a special token to the vocabulary simply gives it a fixed ID. Actually placing them at the beginning and end of samples is typically done by data processing code or chat templates -- the tokenizer does not automatically insert them during encoding.

## 4. Training a Real Tokenizer

In [12]:
# Prepare training corpus
# Use Tiny Shakespeare corpus (public domain, about 1MB of plain text)
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
print("Downloading Tiny Shakespeare corpus...")
raw_text = urllib.request.urlopen(url).read().decode("utf-8")
train_lines = [line for line in raw_text.split("\n") if line.strip()]

# Corpus statistics
total_chars = len(raw_text)
unique_chars = len(set(raw_text))
avg_line_len = total_chars // len(train_lines)

print(f"Download complete. Corpus statistics:")
print(f"  Total lines:       {len(train_lines):,}")
print(f"  Total characters:  {total_chars:,}")
print(f"  Unique characters: {unique_chars}")
print(f"  Average line length: {avg_line_len} characters")
print()
print(f"First 5 lines:")
for i, line in enumerate(train_lines[:5]):
    print(f"  [{i}] {line}")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Total characters:Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.  [0] First Citizen:
  [1] Before we proceed any further, hear me speak.
  [2] All:
  [3] Speak, speak.
  [4] First Citizen:


**Configure and Train**

When training a real tokenizer, you simultaneously decide the vocabulary size, pre-tokenizer, special tokens, and the strategy for handling unknown characters. Below we use the `tokenizers` library to train a BPE tokenizer with a vocabulary size of 4000.

In [13]:
# Configure parameters and train
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCAB_SIZE = 4000

my_tokenizer = Tokenizer(BPE(unk_token="<unk>"))
my_tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<unk>", "<s>", "</s>"],
    show_progress=True,
)

print(f"Training config: vocab={VOCAB_SIZE}, pre-tokenizer=Whitespace, special_tokens=['<unk>','<s>','</s>']")
print("Starting training...")
my_tokenizer.train_from_iterator(train_lines, trainer=trainer)
print(f"Training complete! Vocabulary size: {my_tokenizer.get_vocab_size()} tokens")

# Demo: special tokens are not automatically inserted
sample_text = "To be, or not to be"
plain = my_tokenizer.encode(sample_text)
bos_id = my_tokenizer.token_to_id("<s>")
eos_id = my_tokenizer.token_to_id("</s>")
print(f"\nOriginal text: {sample_text!r}")
print(f"Plain encode: {plain.tokens}")
print(f"Wrapped with BOS/EOS: {['<s>'] + plain.tokens + ['</s>']}")
print(f"Key point: <s> and </s> have fixed IDs, but encode doesn't automatically add them")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Original text: 'To be, or not to be'Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

**Inspect What the Tokenizer Learned**

After training, don't just look at the vocabulary size number. We should also check the token length distribution, single-character coverage, common subwords, and long tokens.

In [14]:
# Inspect what the tokenizer learned
vocab = my_tokenizer.get_vocab()

# Token length distribution
length_dist = {}
for token in vocab:
    length = len(token)
    if length not in length_dist:
        length_dist[length] = 0
    length_dist[length] += 1

print("=== Token length distribution in vocabulary ===")
for length in sorted(length_dist):
    bar = "#" * (length_dist[length] // 5)
    print(f"  Length {length:2d}: {length_dist[length]:>4} tokens  {bar}")

# Longer tokens
long_tokens = sorted([t for t in vocab if len(t) >= 5], key=len, reverse=True)
print(f"\nTokens with length >= 5 (total {len(long_tokens)}, showing first 15):")
for t in long_tokens[:15]:
    print(f"  '{t}' (id={vocab[t]})")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed 

### How to Judge Whether a Tokenizer Is Good

After training a tokenizer, you need to check its quality from several angles:

1. **Compression ratio**: How many characters each token covers on average. Higher compression means shorter sequences. GPT-2 achieves about 3.5 characters/token on English, but only about 1.5 on Chinese.
2. **Decoding consistency**: After encoding and then decoding, you want to recover the original text. If `<unk>` has already been used, the original characters are lost and cannot be recovered.
3. **Coverage**: When encountering text not seen during training, it should not simply error out. Byte-level tokenizers have the highest coverage -- 256 bytes can represent any Unicode text.
4. **Token semantic quality**: The resulting tokens should be reasonable linguistic units. Keeping "the" as a whole is better than splitting it into "th" + "e".

In [15]:
# Evaluate our tokenizer against four criteria

# ---- Metric 1 & 4: Compression ratio + token semantic quality ----
# For the same sentence, check whether tokens are split reasonably and what the compression ratio is
test_cases = [
    "To be, or not to be, that is the question",
    "The king is dead",
    "Hello world",
    "327 + 1 = 328",
    "xyz",  # word not in training corpus
]

print("=== 1. Compression ratio & token quality ===")
print("Good tokenizer: common words kept intact, rare words split into known subwords, no cross-word merging")
print()

for text in test_cases:
    encoding = my_tokenizer.encode(text)
    ratio = len(text) / len(encoding.ids) if len(encoding.ids) > 0 else 0
    print(f"'{text}'")
    print(f"  {len(encoding.ids)} tokens, compression ratio {ratio:.1f}x")
    print(f"  {encoding.tokens}")
    print()

# ---- Metric 2: Decoding consistency ----
print("=== 2. Decoding consistency ===")
print("encode -> decode should ideally recover the original text; if <unk> appears, information is already lost.")
print("Also, different pre-tokenizer / decoder combinations affect space and punctuation restoration.")
print()
for text in test_cases:
    encoding = my_tokenizer.encode(text)
    decoded = my_tokenizer.decode(encoding.ids)
    ok = "OK" if decoded == text else "FAIL"
    print(f"  [{ok}] '{text}' -> '{decoded}'")

# ---- Metric 3: Coverage ----
print()
print("=== 3. Coverage ===")
print("When encountering characters not seen in the corpus, a character-level tokenizer uses <unk> as replacement.")
print("A byte-level tokenizer (like GPT-2) doesn't have this problem.")
coverage_tests = ["Hello world", "xyz", "中文"]
for text in coverage_tests:
    encoding = my_tokenizer.encode(text)
    has_unk = "<unk>" in encoding.tokens
    status = "has <unk> (character not in training corpus)" if has_unk else "all tokens in vocabulary"
    print(f"  '{text}' -> {status}")

# ---- Overall compression effect ----
print()
print("=== Overall compression effect ===")
# Sample first 10000 characters for a simple statistical check; real projects test on held-out validation sets
sample = raw_text[:10000]
enc = my_tokenizer.encode(sample)
ratio = len(sample) / len(enc.ids)
print(f"  First 10000 characters: {len(enc.ids):,} tokens, compression ratio {ratio:.2f}x")
print(f"  Decoding consistency: {'OK' if my_tokenizer.decode(enc.ids) == sample else 'FAIL'}")
print()
print(f"Reference: GPT-2 compression ratio on English is about 3.5x")
print(f"Our tokenizer achieves {ratio:.1f}x on Tiny Shakespeare,")
print(f"The gap comes mainly from vocabulary size (4000 vs 50257) and lack of byte-level starting point")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
'To be, or not to be, that is the question'
Read the values printed above and connect them to the concept in this cell.  ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question']

'The king is dead'
Read the values printed above and connect them to the concept in this cell.  ['The', 'king', 'is', 'dead']

'Hello world'
Read the values printed above and connect them to the concept in this cell.  ['He', 'llo', 'world']

'327 + 1 = 328'
Read the values printed above and connect them to the concept in this cell.  ['3', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '3', '<unk>', '<unk>']

'xyz'
Read the values printed above and connect them to the concept in this cell.  ['x', 'y', 'z']

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept

**How Vocabulary Size Affects Compression**

Finally, let's look at `vocab_size` in isolation: as the vocabulary grows, common fragments get merged more completely, but the returns don't keep growing linearly.

Also note: `vocab_size` is typically a target or upper bound, not a mathematical guarantee. If the corpus is too small and there aren't enough pairs to merge, the final vocabulary may be smaller; if the initial alphabet and special tokens already exceed it, the final vocabulary won't be forcibly trimmed.

In [16]:
# Compare how different vocabulary sizes affect compression
# Train and test with the same sentence at different vocab sizes

test_text = "To be, or not to be, that is the question"

print(f"Test text: '{test_text}'")
print(f"Original text characters: {len(test_text)}")
print()
print(f"{'Vocab size':>8} | {'Tokens':>8} | {'Ratio':>6} | Tokenization result")
print("-" * 72)

for vs in [500, 1000, 2000, 4000]:
    t = Tokenizer(BPE(unk_token="<unk>"))
    t.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vs,
        special_tokens=["<unk>", "<s>", "</s>"],
    )
    t.train_from_iterator(train_lines, trainer=trainer)

    enc = t.encode(test_text)
    ratio = len(test_text) / len(enc.ids) if len(enc.ids) > 0 else 0
    actual = t.get_vocab_size()

    print(f"{actual:>8} | {len(enc.ids):>8} | {ratio:>5.1f}x | {enc.tokens}")

print()
print("Observations:")
print("  - Vocab 500 -> 1000: compression improves significantly (token count drops noticeably)")
print("  - Vocab 2000 -> 4000: improvement diminishes (common words already merged, new merges are rare combinations)")
print("  - Industry chooses 32K-100K because corpora are larger and more multilingual, needing more subwords for coverage")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Number of tokens: ------------------------------------------------------------------------
     500 |       14 |   2.9x | ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'que', 'st', 'ion']
    1000 |       14 |   2.9x | ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'que', 'st', 'ion']
    2000 |       13 |   3.2x | ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'quest', 'ion']
    4000 |       12 |   3.4x | ['To', 'be', ',', 'or', 'not', 'to', 'be', ',', 'that', 'is', 'the', 'question']

observe:  - Number of tokens:   - Read the values printed above and connect them to the concept in this cell.  - Read the values printed above and connect them to the concept in this cell.

### Vocabulary Sizes in Practice

Our mini BPE vocabulary has only 35 tokens. Industrial choices are much larger:

| Tool/Library | Representative Model | Vocabulary Size | Starting Point |
|:---|:---|:---|:---|
| tiktoken | GPT-2 | 50,257 | Bytes |
| tiktoken | GPT-4 | 100,256 | Bytes |
| SentencePiece | LLaMA | 32,000 | Bytes |

Too small a vocabulary (thousands): sequences become very long, and Self-Attention computation scales with the square of sequence length. Too large (millions): the Embedding layer parameters balloon -- vocabulary size times dimension is the Embedding matrix size. At the same time, many tokens appear only a few times, making them hard for the model to learn.

32K-100K is the practical sweet spot. Multilingual models tend toward larger vocabularies; GPT-4 expanded its vocabulary from 50K to 100K primarily to better support non-English text.

#### Appendix: Special Tokens and the Vocabulary

In practice, a Tokenizer also uses certain manually defined control symbols in addition to the tokens derived from text. These are called special tokens. The three most common ones are: `<BOS>` (Beginning of Sequence) to mark the start of a sequence, `<EOS>` (End of Sequence) to mark the end, and `<PAD>` (Padding) to pad variable-length sequences to the same length.

These special tokens have their own fixed IDs in the vocabulary, and they look no different from regular tokens -- they are all just numbers. However, there are a few ways in which they are special.

First, special tokens are not produced through BPE's statistical merging. Regular tokens are built by the tokenizer counting high-frequency pairs in the corpus and merging them step by step; special tokens, on the other hand, are manually specified when training the tokenizer -- their positions are reserved before training begins, and they are hard-coded into the vocabulary without participating in any statistical process.

Second, the tokenizer does not automatically insert special tokens during encode. For example, if you give "hello world" to the tokenizer, it only returns the IDs for "hello" and "world" -- it will not automatically prepend a `<BOS>` ID or append an `<EOS>` ID. Whether to add these symbols and where to place them is decided by whoever prepares the training data. For instance, in preprocessing code, each sentence might be manually wrapped as `[<BOS> ID] + text IDs + [<EOS> ID]` before being fed to the model. The tokenizer only manages "what ID each token maps to" -- it does not decide "which tokens this sentence should have". The latter is determined by the chat template: a set of rules specifying what to add before the user's message, what to add before and after the model's response, and so on. This operates outside the tokenizer.

Third, special tokens are handled differently during training and inference. `<PAD>` is merely a placeholder; when computing Attention, the model uses an attention mask to zero out its positions, and when computing the loss, it is similarly skipped so that padding does not interfere with real content. `<EOS>` serves as a stop signal during text generation -- when the model generates tokens one by one, decoding stops immediately once the `<EOS>` ID is produced.

The embeddings of these special tokens participate in training and gradient updates just like other parameters; they are only selectively excluded in different computational stages. More details will be covered when we implement Mini-GPT later.

## Summary

What we learned in this section:

- BPE starts from characters and repeatedly executes the loop: "count pairs -> merge most frequent -> record rule"
- Merge rules are ordered; encoding new text replays them in order
- BPE handles most unknown words -- as long as the characters are known, new words naturally fall back to subwords or characters
- Industrial BPE adds pre-tokenization (preventing cross-word merging) and byte-level starting point (covering all Unicode)
- Vocabulary size balances compression ratio against parameter count; mainstream range is 32K-100K
- Special tokens (BOS / EOS / UNK) are not learned by BPE; they are control signals manually added to the vocabulary
- Step-by-step derivation of the BPE algorithm is in the appendix at the end of this chapter

Next section moves to Embedding -- token IDs are just numbers; the neural network needs to convert them into vectors before it can compute.

## Exercises

Three exercises covering two topics:

1. BPE repeatedly counts adjacent pairs and merges the most frequent pair.
2. Real tokenizers often start from bytes and encode using ordered merge rules.

> **About AI assistance**: You can ask AI for hints, step breakdowns, or direction checks, but it's not recommended to have AI complete the exercises outright. The key point of BPE is seeing for yourself how tokens gradually grow.

### Exercise 1: Counting Pairs

The first step of BPE is counting the frequency of adjacent token pairs.

**Hint**: A pair is `(tokens[i], tokens[i + 1])`.

In [ ]:
# Exercise 1: Count adjacent pairs - fill in the blanks

tokens = list("banana")
pair_counts = {}

for i in range(len(tokens) - 1):
    pair = (tokens[i], tokens[i + 1])
    # TODO: Replace the triple-quoted content below with your code
    """Update pair_counts[pair] here"""

assert pair_counts[("a", "n")] == 2, dict(pair_counts)
assert pair_counts[("n", "a")] == 2, dict(pair_counts)
assert pair_counts[("b", "a")] == 1, dict(pair_counts)
print("✅ Exercise 1 passed: you've learned that BPE's first step is counting adjacent pairs")

### Exercise 2: Merging Pairs

After finding the most frequent pair, merge it into a new token.

**Hint**: When the pair matches, consume two tokens at once; otherwise, consume only the current token.

In [ ]:
# Exercise 2: Merge pairs - fill in the blanks
def merge_pair(tokens, pair):
    """Merge adjacent tokens matching the given pair into a new token"""
    merged = []
    i = 0
    while i < len(tokens):
        # TODO: Replace the triple-quoted content below with your code
        """Check if the pair matches here, and update merged and i"""
    return merged

result = merge_pair(list("banana"), ("a", "n"))

assert result == ["b", "an", "an", "a"], result
print("✅ Exercise 2 passed: you've learned that BPE's core action is merging")

### Exercise 3: Byte-Level Starting Point

Modern tokenizers often start from bytes, so any Unicode character can be handled.

**Hint**: In Python, you can use `text.encode("utf-8")` to see the bytes corresponding to a string.

In [ ]:
# Exercise 3: UTF-8 bytes - fill in the blanks
text = "你😊"

# TODO: Replace the triple-quoted content below with your code
byte_values = """Encode text to a list of UTF-8 byte values here"""

assert not isinstance(byte_values, str), "Please replace the triple-quoted placeholder first"
assert byte_values == [228, 189, 160, 240, 159, 152, 138], byte_values
print("✅ Exercise 3 passed: you understand why modern tokenizers often start from bytes")

## Appendix: Step-by-Step BPE Derivation

> The following appendix is for readers who want to understand every step of BPE in detail. Starting from a character-level baseline, we manually count pairs, merge, and train, gradually building a mini BPE tokenizer.

The `BPETokenizer` class and `corpus` variable have already been defined in the main text. The code in this appendix can use them directly.

### A.1 Starting Point: Character-Level Splitting

BPE starts at the character level: first split text into characters, where each character is a token. The reason for starting from characters is that while they are granular, they are stable -- as long as the characters are in the vocabulary, new words can still be split apart and represented.

One detail: how should spaces be handled? Real GPT tokenizers do more sophisticated processing of spaces. This mini version treats spaces as ordinary characters, so we can clearly see that BPE doesn't only merge letters -- it might also merge combinations like `e` and a space.

In [17]:
# Split all sentences into character lists
def text_to_tokens(text):
    """Split a piece of text into a character-level token list"""
    return list(text)

# Look at the first sentence
sentence = corpus[0]
chars = text_to_tokens(sentence)
print(f"Original text:   '{sentence}'")
print(f"Split into characters: {chars}")
print(f"Total {len(chars)} tokens")

# Split every sentence in the corpus into character lists
corpus_tokens = [text_to_tokens(s) for s in corpus]
print(f"\nCharacter count per sentence in corpus: {[len(t) for t in corpus_tokens]}")

Original text: 'the cat sat on the mat'Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

### A.2 Counting Adjacent Pairs

Next, count the frequency of all adjacent token pairs across the entire corpus. The one that appears most often is the best candidate for merging first.

Using a short sequence as an example:

```
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ^--^   ^--^   ^--^   ^--^   ^--^
 ('t','h') ('h','e') ('e',' ') (' ','c') ('c','a') ('a','t')
```

A single sentence doesn't reveal much about patterns, so we need to count across the entire corpus.

In [18]:
def count_pairs(token_lists):
    """
    Count the frequency of all adjacent token pairs across the entire corpus

    Parameters:
        token_lists: List[List[str]], each sentence is a character list
    Returns:
        dict: {(token_a, token_b): occurrence count}
    """
    pairs = {}
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            if pair not in pairs:
                pairs[pair] = 0
            pairs[pair] += 1
    return pairs

# Count pair frequencies in the initial state (all character-level)
initial_pairs = count_pairs(corpus_tokens)

print("=== All adjacent pair frequencies in initial state ===")
for pair, count in sorted(initial_pairs.items(), key=lambda x: -x[1]):
    print(f"  {pair}: {count} times")

print(f"\nMost frequent pair: {max(initial_pairs, key=initial_pairs.get)}")
print(f"Appeared {initial_pairs[max(initial_pairs, key=initial_pairs.get)]} times")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed a

### A.3 Merging the Most Frequent Pair

After finding the most frequent pair, BPE merges all occurrences of that pair in the corpus:

```
Before merge: ['t', 'h', 'e', ' ', 'c', 'a', 't']
Suppose we merge: ('t', 'h')
After merge:  ['th', 'e', ' ', 'c', 'a', 't']
                ^ 't' and 'h' become one token 'th'
```

After merging, `th` participates as a whole in subsequent counting. The next time we count pairs, we might see new combinations like `('th', 'e')`.

In [19]:
def merge_pair(token_lists, pair_to_merge):
    """
    Merge all occurrences of the specified pair in the corpus into one token

    Parameters:
        token_lists: List[List[str]], current token representation of the corpus
        pair_to_merge: (str, str), the pair to merge
    Returns:
        Merged new token_lists, new token
    """
    a, b = pair_to_merge
    new_token = a + b

    new_token_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_token_lists.append(new_tokens)

    return new_token_lists, new_token

# Demo: perform one merge
best_pair = max(initial_pairs, key=initial_pairs.get)
print(f"Pair to merge: {best_pair}")

merged_tokens, new_token = merge_pair(corpus_tokens, best_pair)
print(f"New token: '{new_token}'")
print(f"\nToken sequence for sentence 0 after merge:")
print(f"  {merged_tokens[0]}")

Read the values printed above and connect them to the concept in this cell.new token: 'e '
Read the values printed above and connect them to the concept in this cell.  ['t', 'h', 'e ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'm', 'a', 't']


### A.4 After Merging, the Statistics Change Too

Merging doesn't just change the vocabulary -- it changes the basis for subsequent statistics. The BPE training loop has only 4 steps:

1. Count pair frequencies
2. Find the most frequent pair
3. Merge that pair
4. Record this merge rule

Then return to step 1. Each merge creates new adjacent relationships -- for example, after getting `th`, we might later get `the`. So BPE doesn't recognize a word all at once; it grows step by step.

### A.5 How the Vocabulary Grows: Only Additions, No Deletions

This is a common source of confusion: after a merge, do old tokens disappear from the vocabulary? The answer is **no**.

What actually changes are two things:
1. The token sequences in the current training corpus get merged, e.g., `['a', 'n']` becomes `['an']`
2. The vocabulary gains a new token, e.g., `'an'` is added

The original `'a'` and `'n'` stay in the vocabulary. Why? Because when encountering other new words later, we might still need to fall back to these smaller units. Let's observe with a minimal example.

In [20]:
# Observe how vocab grows step by step
toy_token_lists = [list("banana")]
toy_vocab = set(toy_token_lists[0])


def print_vocab_state(step, token_lists, vocab, new_token=None):
    """Print current corpus token sequences and current vocabulary"""
    title = f"Step {step}"
    if new_token is not None:
        title += f" | New token: {new_token!r}"
    print(title)
    print(f"  Current corpus tokens: {token_lists[0]}")
    print(f"  Current vocab({len(vocab)}): {sorted(vocab)}")
    print()


print_vocab_state(0, toy_token_lists, toy_vocab)

for step in range(1, 4):
    pair_counts = count_pairs(toy_token_lists)
    best_pair = max(pair_counts, key=pair_counts.get)
    best_count = pair_counts[best_pair]

    toy_token_lists, new_token = merge_pair(toy_token_lists, best_pair)
    toy_vocab.add(new_token)

    print(f"Most frequent pair this round: {best_pair}, appeared {best_count} times")
    print_vocab_state(step, toy_token_lists, toy_vocab, new_token)

print("Key observation: vocab only appends new tokens; old characters remain in the vocabulary.")

Step 0
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values

### A.6 Complete Training Loop

Now let's run the full loop. There's one important parameter: `num_merges` -- the number of merge operations. The number of merges determines the vocabulary size: each merge adds one new token. Let's set `num_merges=15` and turn on verbose mode to observe each step.

In [21]:
# Retrain using the BPETokenizer class defined in the main text, with verbose output
bpe_verbose = BPETokenizer()
_ = bpe_verbose.train(corpus, num_merges=15, verbose=True)

Read the values printed above and connect them to the concept in this cell.============================================================
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and conn

### A.7 Reading the Training Log

Looking back at the training log above, it's a gradual growth process:

```
Step 1: ('e', ' ')   -> 'e '     <- 'e' followed by space is very common
Step 2: ('t', 'h')   -> 'th'     <- 'the' appears often, so 'th' shows up first
Step 3: ('th', 'e ') -> 'the '   <- 'th' and 'e ' continue to merge
Step 4: ('a', 't')   -> 'at'     <- cat / sat / mat make 'at' very common
Step 5: ('o', 'g')   -> 'og'     <- 'og' in dog / log also gets merged
```

BPE is not a program that "recognizes English words." It simply does statistics: whichever things are next to each other most often gets merged first. This also explains why vocabulary size is controllable: 15 merges means only 15 new tokens.

### How Merge Rules Grow Step by Step

Common combinations appear first, then longer combinations continue to grow on top of them. Below is the complete merge rules list -- observe how `the` is assembled step by step.

In [22]:
print("Complete Merge Rules list (in training order):")
print("Watch how 'the' is born:")
print()

for i, (a, b) in enumerate(bpe_verbose.merge_rules):
    arrow = ""
    merged = a + b
    if merged in ['th', 'the ', 'the c', 'the cat ']:
        arrow = "  <- 'the' birth process!"
    if a in ['th', 'the ', 'the c'] or b in ['e ', 'c', 'at ']:
        arrow = "  <- 'the' birth process!"
    print(f"  Rule {i+1:2d}: '{a}' + '{b}' -> '{merged}'{arrow}")

print()
print("Key observation: 'the ' was not recognized all at once.")
print("It grew from 't'+'h'->'th', then merged with 'e ', gradually forming.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
  Rule  1: 'e' + ' ' → 'e '
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  Rule  4: 'a' + 't' → 'at'
  Rule  5: 'o' + 'g' → 'og'
  Rule  6: 'at' + ' ' → 'at '
  Rule  7: 's' + ' ' → 's '
  Rule  8: 'd' + 'og' → 'dog'
  Rule  9: 'i' + 's ' → 'is '
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  Rule 12: ' ' + 'the ' → ' the '
  Rule 13: 'n' + 'd' → 'nd'
Read the values printed above and connect them to the concept in this cell.  Rule 15: 'sat ' + 'o' → 'sat o'

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### A.8 Greedy Merging in Encode

After training, we have a sequence of merge rules. Now when new text arrives, how do we encode it?

The process is straightforward:
1. First split the new text into characters
2. Apply each merge rule from training, one at a time, trying to merge
3. Finally look up the remaining tokens to get IDs

```
"the cat"
  -> ['t','h','e',' ','c','a','t']
  -> Apply rule 1: ('e',' ') -> 'e '
  -> ['t','h','e ','c','a','t']
  -> Apply rule 2: ('t','h') -> 'th'
  -> ['th','e ','c','a','t']
  -> Apply rule 3: ('th','e ') -> 'the '
  -> ['the ','c','a','t']
  -> Apply rule 4: ('a','t') -> 'at'
  -> ['the ','c','at']
```

The most important point here: **order matters.** Because what gets merged first affects what can be merged later. Encode doesn't re-count frequencies; it follows the rules learned during training.

### A.9 From Character-Level to Byte-Level: GPT-2 Style BPE

GPT-2's Tokenizer is essentially a byte-level BPE tokenizer. Compared to our mini version, it has three upgrades:

1. **Starting point switched from characters to bytes**: 256 bytes form the base vocabulary, covering all Unicode
2. **Added pre-tokenization**: Uses regex to first split text into independent fragments, preventing cross-word merging
3. **Trains merge rules**: Performs BPE on pre-tokenized fragments, merging byte pairs

GPT-2's final vocabulary size = 256 (base bytes) + 50,000 (learned merges) + 1 (special token) = 50,257. Below we use the `tokenizers` library on a small corpus to demonstrate how the vocabulary grows from 256 bytes.

In [23]:
# Reproduce GPT-2 Tokenizer: byte-level BPE
# Steps: 256 base bytes -> pre-tokenization -> BPE training -> observe vocab growth
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.pre_tokenizers import ByteLevel
    from tokenizers.trainers import BpeTrainer

    # GPT-2's training corpus (using a small corpus here for demo; real GPT-2 used massive web text)
    gpt2_style_corpus = [
        "the cat sat on the mat",
        "the dog sat on the log",
        "the cat and the dog",
        "hello GPT-2 style BPE",
        "numbers: 327 + 1 = 328",
        "unicode: 你好 😊",
    ]

    # Step 1: Base vocabulary = 256 bytes
    # ByteLevel.alphabet() returns these 256 byte tokens
    base_byte_tokens = set(ByteLevel.alphabet())
    special_tokens = ["HUMAN>"]  # GPT-2 has only one special token

    print(f"Step 1: Base vocabulary = {len(base_byte_tokens)} bytes")
    print(f"Special token: {special_tokens}")
    print(f"Note: The 'Ġ' in the output represents a leading space, which is the result of ByteLevel encoding.")
    print()

    # Step 2: Gradually increase vocab_size and observe what new tokens BPE learns
    previous_learned_tokens = set()

    for target_vocab_size in [260, 268, 276]:
        # Create an empty BPE model (no unk_token specified, because byte-level never encounters unknown characters)
        tokenizer = Tokenizer(BPE())

        # Add ByteLevel pre-tokenizer: handles spaces and Unicode
        # This is one of GPT-2's core features: first convert text to byte sequences, then apply BPE
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

        # Configure trainer
        # initial_alphabet: 256 bytes as starting point
        # vocab_size: target vocabulary size = 256 + number of merges
        trainer = BpeTrainer(
            vocab_size=target_vocab_size,
            initial_alphabet=ByteLevel.alphabet(),
            special_tokens=special_tokens,
            show_progress=False,
        )
        tokenizer.train_from_iterator(gpt2_style_corpus, trainer=trainer)

        # Count new tokens learned this round
        vocab = tokenizer.get_vocab()
        learned_tokens = set(vocab) - base_byte_tokens - set(special_tokens)
        new_tokens = learned_tokens - previous_learned_tokens

        learned_sorted = sorted(learned_tokens, key=lambda token: vocab[token])
        new_sorted = sorted(new_tokens, key=lambda token: vocab[token])

        print(f"target vocab_size={target_vocab_size}, actual size={len(vocab)}")
        print(f"  BPE new tokens: {len(learned_sorted)}")
        print(f"  Newly learned this round: {new_sorted[:12]}")
        print(f"  Cumulative new tokens (first 12): {learned_sorted[:12]}")
        print()

        previous_learned_tokens = learned_tokens

    print("Observations:")
    print("  - Initial 256 bytes + new merge tokens each round = final vocab")
    print("  - Tokens starting with 'Ġ' represent subwords with leading spaces (e.g., 'Ġthe' = ' the')")
    print("  - Real GPT-2 training did 50,000 merges, final vocab = 50,257")

except Exception as e:
    print("Could not run byte-level BPE demo.")
    print(f"Reason: {e}")
    print("Please install tokenizers: pip install tokenizers")

Read the values printed above and connect them to the concept in this cell.Special token: ['HUMAN>']
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Number of tokens: Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Number of tokens: Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Number of tokens: Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
observe:  - Read the values printed above and connect them to the concept in this cell.  - 

### A.10 Vocabulary Sizes in Practice

| Tool/Library | Representative Model | Vocabulary Size | Starting Point |
|:---|:---|:---|:---|
| tiktoken | GPT-2 | 50,257 | Bytes |
| tiktoken | GPT-4 | 100,256 | Bytes |
| SentencePiece | LLaMA | 32,000 | Bytes |

Too small a vocabulary (thousands): sequences become very long, and Self-Attention computation scales with the square of sequence length. Too large (millions): the Embedding layer parameters balloon, and many tokens appear only a few times, making them hard for the model to learn. 32K-100K is the practical sweet spot.